<a href="https://colab.research.google.com/github/fickas/crab_project/blob/main/notebooks/salt_marsh_crab_wellfleet_production1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#What is needed for production

What the production output actually is. Pixel-level IoU is a means to your end, not the end itself. Your real deliverable is polygons around crab-affected regions. The conversion path from trained model to polygons is roughly:

1. Predict per-pixel softmax probabilities over the full orthomosaic via sliding-window inference with overlap blending.
2. Take argmax for class predictions; separately keep max-probability for confidence.
3. Apply a confidence threshold (e.g., max softmax ≥ 0.7) to drop low-confidence pixels.
4. Binarize or keep multi-class (pixels in classes-of-interest → 1, else → 0; or keep per-class masks).
5. Morphology cleanup: dilate-then-erode to close small gaps, erode-then-dilate to remove noise specks, minimum area threshold to drop fragments.
6. Connected components → individual regions.
7. Vectorize with rasterio.features.shapes → polygon geometries.
8. Save as a shapefile with class and confidence attributes.

In [ ]:
# Cell 1 — repo + dependencies
!git clone https://github.com/fickas/crab_project.git /content/marsh-crab 2>/dev/null || \
 (cd /content/marsh-crab && git pull)
!pip install -r /content/marsh-crab/requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.0 MB/s eta 0:00:00


#Wait until marsh-crab shows up locally

In [ ]:
# Cell 2 — Python setup
import sys
sys.path.insert(0, '/content/marsh-crab')
import marsh_utils as mu

##Run on changes

In [ ]:
# Run this cell whenever you edit the repo
!cd /content/marsh-crab && git pull
import importlib
importlib.reload(mu)

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 980 bytes | 163.00 KiB/s, done.
From https://github.com/fickas/crab_project
   d703015..ea8cfe3  main       -> origin/main
Updating d703015..ea8cfe3
Fast-forward
 marsh_utils.py | 1 +
 1 file changed, 1 insertion(+)


<module 'marsh_utils' from '/content/marsh-crab/marsh_utils.py'>

##Bring in results of training

In [ ]:
!pip install segmentation-models-pytorch

import os
import json
import torch
import segmentation_models_pytorch as smp


def load_training_artifacts(artifact_dir, device='cuda'):
    with open(os.path.join(artifact_dir, 'config.json')) as f:
        bundle = json.load(f)

    class Config:  # fresh shell
        pass
    for k, v in bundle['config'].items():
        print(f'loading {k} with value {v}')
        setattr(Config, k, v)

    # JSON loses tuples and int dict keys — restore them
    Config.BAND_SPEC = [tuple(item) for item in Config.BAND_SPEC]
    Config.CLASS_NAMES = {int(k): v for k, v in Config.CLASS_NAMES.items()}
    Config.CONFIDENCE_THRESHOLDS = {
        int(k): float(v) for k,v in Config.CONFIDENCE_THRESHOLDS.items()
    }

    model = smp.Unet(
        encoder_name=Config.ENCODER,
        encoder_weights=None,
        in_channels=len(Config.BAND_SPEC),
        classes=len(Config.CLASS_NAMES),
    )
    state_dict = torch.load(os.path.join(artifact_dir, 'model.pt'), map_location=device)
    model.load_state_dict(state_dict)
    model.to(device).eval()

    channel_means = np.array(bundle['normalization']['mean'])
    channel_stds  = np.array(bundle['normalization']['std'])

    return model, Config, channel_means, channel_stds, bundle.get('training_summary', {})

In [ ]:
run_name = 'runs/run_2026-06-09_18-28-34/'     #{datetime.now():%Y-%m-%d_%H-%M-%S}'

In [ ]:
synthetic = True

In [ ]:
# 1. Imports — minimal, no training-only stuff
import os
import json
import numpy as np
import torch
import rasterio
import geopandas as gpd
import segmentation_models_pytorch as smp

# Plus your project's utility modules (build_patches_with_splits_multi if needed for inference,
# ensure_indices, etc.) — only the ones needed downstream from a trained model

# 2. Paths for production
base = '/content/drive/MyDrive/salt_marsh/crab_project/WEL'
paths = {
    'pan_orthomosaic': f'{base}/synthetic_test/wellfleet_high_res_synthetic/pan.tif' if synthetic else f'{base}/wellfleet_high_res/pan.tif',
    'pansharp_ms':     f'{base}/synthetic_test/wellfleet_high_res_synthetic/pansharp_5band.tif' if synthetic else f'{base}/wellfleet_high_res/pansharp_5band.tif',
    'crab_polygons':   f'{base}/synthetic_test/wellfleet_high_res_synthetic/crab_polygons.shp' if synthetic else f'{base}/wellfleet_high_res/crab_polygons.shp',
    'artifacts':       f'{base}/synthetic_test/wellfleet_high_res_synthetic/{run_name}/' if synthetic else f'{base}/{run_name}/',  #from training notebook,
    'output_polygons': f'{base}/synthetic_test/wellfleet_high_res_synthetic/predictions/crab_polygons_predicted.shp' if synthetic else f'{base}/predictions/crab_polygons_predicted.shp',
}

# 3. Ensure derived bands exist (idempotent — skips if already cached)
paths = mu.ensure_indices(paths)

# 4. Load model and config
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, config, channel_means, channel_stds, training_summary = load_training_artifacts(paths['artifacts'], device=device)

print(f"Loaded model from {paths['artifacts']}")
print(f"Best val mIoU at training: {training_summary.get('best_val_miou', 'unknown'):.4f}")
print(f"Band spec: {config.BAND_SPEC}")
print(f"Classes:   {config.CLASS_NAMES}")


NDVI:
  exists, skipping: /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic/ndvi.tif
NDRE:
  exists, skipping: /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic/ndre.tif
loading CLASS_COLUMN with value Class
loading CLASS_NAMES with value {'0': 'other', '1': 'healthy_bank', '2': 'eroding_non_crab', '3': 'crab_edge', '4': 'crab_platform', '5': 'collapsed'}
loading CLASSES with value {'other': 1, 'healthy_bank': 2, 'eroding_non_crab': 3, 'crab_edge': 4, 'crab_platform': 5, 'collapsed': 6}
loading CLASSES_OF_INTEREST with value [3, 4, 5]
loading PRIORITY with value [5, 4, 3, 2, 1, 0]
loading IGNORE_INDEX with value 255
loading QGIS_TO_MODEL with value {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5}
loading N_CLASSES with value 6
loading SEED with value 42
loading RESOLUTION_CM with value 1.0
loading BAND_SPEC with value [['pan_orthomosaic', 1], ['ndvi', 1], ['ndre', 1]]
loading PATCH_SIZE with value

#sliding-window inference

In [ ]:
from datetime import datetime

date_name = f'{datetime.now():%Y-%m-%d_%H-%M-%S}'

In [ ]:

output_path=f'{base}/synthetic_test/wellfleet_high_res_synthetic/model1_predictions/full_probs_{date_name}.tif' if synthetic else f'{base}/wellfleet_high_res/model1_predictions/probs_{date_name}.tif'


In [ ]:
mu.predict_full_raster(
    model=model,
    cfg=config,
    paths=paths,
    channel_means=channel_means,
    channel_stds=channel_stds,
    output_path=output_path,
    device=device,
)

Inference grid: 6000×6000, 529 patches (512×512, stride 256)
  8/529 patches
  408/529 patches
Wrote predictions: /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic/predictions/full_probs_2026-06-09_19-55-58.tif
  shape: (6, 6000, 6000)  classes: ['other', 'healthy_bank', 'eroding_non_crab', 'crab_edge', 'crab_platform', 'collapsed']


# Produce shapefiles

## How the parameters affect output:

* min_area_m2=1.0 drops any polygon smaller than 1 m² (at 1cm GSD that's ~10,000 pixels). Reasonable starting value for marsh-scale features. Drop it lower (0.1) if you want to catch small Pattern B clusters; raise it higher (5–10) if you want only Pattern A ribbons.

* morph_close_pixels=3 fills gaps up to 3 pixels = 3cm at 1cm GSD. Smooths over single-pixel "holes" inside a polygon. Bump up to 10+ if your raw predictions are speckled inside what should be solid regions.

* morph_open_pixels=2 removes specks up to 2 pixels. Drops isolated single-pixel false positives. Bump up to 5+ for more aggressive denoising at the cost of losing thin features.

##Two things worth knowing about the output:

1. The mean_confidence field per polygon tells you "how confident was the model on average inside this polygon." Useful for downstream prioritization — if you want to feed only high-confidence polygons to Model 2, filter polygons_gdf by mean_confidence > 0.85 rather than relying solely on the per-class threshold (which only required any pixel to clear the bar, not the polygon average).

2. The CRS comes from the source raster (EPSG:26919 for Wellfleet), so area_m2 is in actual square meters — no conversion needed. If you load these into QGIS alongside the orthomosaic, they'll register correctly.

###Per class thresholds

Each pixel can only belong to one class — the one that wins argmax. Different pixels use different thresholds depending on which class won, but no single pixel is being compared against multiple thresholds. There's no overlap or ambiguity.

The per-pixel decision is three steps:

1. argmax — which class the model thinks is most likely for this pixel (one specific class).
2. max_prob — how confident the model is in that prediction (one value).
3. Include in class c's polygon set if argmax == c AND max_prob >= threshold[c].

In [ ]:
prob_raster_path = f'{base}/synthetic_test/wellfleet_high_res_synthetic/model1_predictions/full_probs_{date_name}.tif' if synthetic else f'{base}/wellfleet_high_res/model1_predictions/full_probs_{date_name}.tif'
output_shp = f'{base}/synthetic_test/wellfleet_high_res_synthetic/model1_predictions/crab_polygons_{date_name}.shp' if synthetic else f'{base}/wellfleet_high_res/model1_predictions/crab_polygons_{date_name}.shp'

In [ ]:
# Step 6 — turn probabilities into polygons
polygons_gdf = mu.predictions_to_polygons(
    prob_raster_path=prob_raster_path,
    cfg=config,
    min_area_m2=1.0,     # drop fragments smaller than 1 m²
)

# Step 7 — save
polygons_gdf.to_file(output_shp)
print(f"Saved {len(polygons_gdf)} polygons to {output_shp}")


Extracted 83 polygons:
               count     sum  mean
class_name                        
collapsed         23   90.24  3.92
crab_edge         34  171.68  5.05
crab_platform     26  109.40  4.21


/tmp/ipykernel_14040/1525871421.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  polygons_gdf.to_file(output_shp)
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'mean_confidence' to 'mean_confi'
  ogr_write(


Saved 83 polygons to /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic/model1_predictions/crab_polygons_2026-06-09_19-55-58.shp


In [ ]:
print(f"\nProduced {len(polygons_gdf)} polygons:")
summary = polygons_gdf.groupby('class_name').agg(
    count=('class', 'count'),
    total_area_m2=('area_m2', 'sum'),
    mean_area_m2=('area_m2', 'mean'),
    mean_conf=('mean_confidence', 'mean'),
).round(2)
print(summary)
print(f"\nTotal area predicted: {polygons_gdf['area_m2'].sum():.1f} m²")
print(f"Output saved to: {output_shp}")


Produced 83 polygons:
               count  total_area_m2  mean_area_m2  mean_conf
class_name                                                  
collapsed         23          90.24          3.92       0.94
crab_edge         34         171.68          5.05       0.97
crab_platform     26         109.40          4.21       0.91

Total area predicted: 371.3 m²
Output saved to: /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic/model1_predictions/crab_polygons_2026-06-09_19-55-58.shp
